# 03 - Agent run on one segment

Runs the fixed LangGraph pipeline on **one segment** of a mined run:

`load_evidence -> diagnose -> redesign -> apply_edits -> validate -> human_review -> finalize`

- `diagnose` and `redesign` are the only LLM steps. The model sees the evidence digest (IDs + code-written summaries), the heuristics catalogue, domain facts and constraints, never events.
- Every LLM output is validated in code (evidence IDs, heuristic IDs, activities, protected activities, trial application of every edit, unsupported numbers) and retried with the issues fed back (max 2 retries).
- `apply_edits` (pure code) applies each proposal's typed edit operations to the as-is process tree and writes `to_be_r<N>_p<k>.bpmn` (+ PNG).
- `validate` (pure code) checks soundness (by construction, structural check, Woflan up to 40 transitions) and computes the replay-based impact **ESTIMATE** (`redesign/simulate.py`; its assumptions are listed with it).
- The run pauses at `human_review` (LangGraph `interrupt()`); this notebook simulates one **reject with comment** (back to `redesign`) and then an **approve**.

**Default: dry run.** `DRY_RUN = True` uses the scripted model in `processpilot.agent.dry_run` (zero API calls, placeholder texts, model id `scripted:dry-run`) on the committed 800-case sample, mined into a temporary folder, so nothing is added to `outputs/`. Set `DRY_RUN = False` for a live run with `PP_MODEL` on the latest mined BPI 2019 run; that spends API quota (2 calls per round, up to 6 with retries). For day-to-day use, the web app runs the same graph.

In [ ]:
import tempfile
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from processpilot.agent.dry_run import DryRunLLM
from processpilot.agent.graph import AgentRunner, load_as_is_tree
from processpilot.agent.report import (
    diagnosis_markdown,
    issue_details,
    outcomes_markdown,
    proposals_markdown,
    validation_log,
    warnings_markdown,
)
from processpilot.agent.runlog import RunLogger
from processpilot.agent.tools import EvidenceTools
from processpilot.config import default_runs_dir, load_env, resolve_source_path
from processpilot.ingest.mapping import load_preset
from processpilot.mining.evidence import EvidenceStore, segment_slug
from processpilot.pipeline import (
    clean_log,
    load_canonical,
    mine_log,
    suggested_cleaning_policy,
    validate_log,
)
from processpilot.process_tree import labels
from processpilot.sources.file_source import FileSource

DRY_RUN = True  # False = live model from PP_MODEL (spends API quota)
SEGMENT = "2-way match"
load_env()

## 1. The mining run

In [ ]:
if DRY_RUN:
    # Mine the committed sample (~10 s) into a temporary folder.
    mapping, preset = load_preset("bpi2019_sample")
    tmp = Path(tempfile.mkdtemp(prefix="pp-nb03-"))
    canonical = load_canonical(
        FileSource(resolve_source_path(preset), cache_dir=tmp / "cache"), mapping
    )
    report = validate_log(canonical, preset)
    clean, _ = clean_log(canonical, report, suggested_cleaning_policy(preset), preset)
    run_dir = tmp / "runs" / "20990101T000000Z"
    mine_log(clean, preset, run_dir)
else:
    # Latest complete mining run (manifest.json is written last).
    _, preset = load_preset("bpi2019")
    run_dir = sorted(p.parent for p in default_runs_dir().glob("*/manifest.json"))[-1]
print("mining run:", run_dir)

## 2. Run to the review interrupt

In [ ]:
if DRY_RUN:
    store = EvidenceStore.from_json(run_dir / "evidence.json")
    model_activities = sorted(labels(load_as_is_tree(run_dir, SEGMENT)))
    llm = DryRunLLM.for_segment(EvidenceTools(store, SEGMENT), model_activities)
else:
    llm = None  # AgentRunner builds the model from PP_MODEL

runner = AgentRunner(run_dir, preset, llm=llm)
print("model:", runner.llm.model_id)
print("constraints:", runner.default_constraints(SEGMENT).model_dump())
agent_run_id, state = runner.start(SEGMENT)
log = RunLogger(run_dir / segment_slug(SEGMENT) / "agent" / agent_run_id)
print("agent run:", agent_run_id, "| status:", state.status, "| error:", state.error)
print("log:", log.directory)

## 3. Diagnosis

In [ ]:
display(Markdown(diagnosis_markdown(state.diagnosis)))

## 4. Proposals (round 1)

In [ ]:
display(Markdown(proposals_markdown(state.proposals)))
display(Markdown(warnings_markdown(state.warnings)))

## 5. Applied edits, soundness and the ESTIMATE

In [ ]:
display(Markdown(outcomes_markdown(state.edit_results, state.validation_results)))

## 6. Validation log (every LLM call, retries, issues)

In [ ]:
display(pd.DataFrame(validation_log(log)))
details = issue_details(log)
pd.DataFrame(details) if details else "no validation issues on any call"

## 7. Analyst rejects with a comment -> back to `redesign`

In [ ]:
COMMENT = (
    "Do not weaken financial controls or segregation of duties. Prefer changes that "
    "reduce waiting before the invoice without merging approval and invoice steps."
)
state = runner.resume(SEGMENT, agent_run_id, {"decision": "reject", "comment": COMMENT})
print("status:", state.status, "| revision:", state.revision_count, "| error:", state.error)
display(Markdown(proposals_markdown(state.proposals)))
display(Markdown(warnings_markdown(state.warnings)))

## 8. Analyst approves -> `finalize`

In [ ]:
state = runner.resume(SEGMENT, agent_run_id, {"decision": "approve", "comment": None})
print("final status:", state.status)
pd.DataFrame([r.model_dump() for r in state.review_history])

## 9. Token usage and cost (from the run log)

Costs are estimates at list price (`configs/agent.yaml`); a dry run reports no tokens and no cost.

In [ ]:
totals = log.usage_totals()
print("served models:", log.served_models())
totals

In [ ]:
sorted(p.name for p in log.directory.iterdir())

In [ ]:
runner.close()